# Phase 2 Agent Pipeline Testing

This notebook is the isolated test bench for the agent, reward, evaluation, and training pipeline layers.

Scope for this notebook:
- `src/env/reward.py`
- `src/evaluation/metrics.py`
- `src/evaluation/evaluate_agent.py`
- `src/agents/sac_agent.py`
- `src/training/train_sac_her.py`

What this notebook does:
- runs deterministic helper-level checks for reward and evaluation logic
- builds the real environment through the training config
- smoke-tests behavioral cloning, replay-buffer seeding, short SAC learning, and prediction
- runs the full smoke training pipeline with isolated output artifacts

What this notebook does not try to prove:
- that the SAC model is well-trained
- that the smoke config beats the greedy baseline
- that longer training runs are stable across all hyperparameters


In [1]:
import json
import sys
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.env.reward import (
    RewardConfig,
    build_step_reward,
    clue_margin,
    compute_goal_conditioned_reward,
    subset_achieved,
)
from src.evaluation.metrics import EpisodeMetrics, summarize_metrics
from src.evaluation.evaluate_agent import evaluate_agent, select_policy_action
from src.agents.bc_pretrain import generate_demonstrations
from src.agents.sac_agent import SACSpymasterAgent, SACTrainingConfig
from src.training.train_sac_her import (
    build_embedding_store,
    build_env_factory,
    load_config,
    run_training_pipeline,
)

pd.set_option("display.max_colwidth", 120)
print(f"Project root: {PROJECT_ROOT}")


Project root: /Users/aeshagandhi/Downloads/MIDS-Sp26/RL/Codenames-Spymaster-RL-fresh


In [2]:
config = load_config(PROJECT_ROOT / "configs" / "smoke_test.yaml")
print(json.dumps({
    "config_name": config["name"],
    "training_total_timesteps": config["training"]["total_timesteps"],
    "bc_demo_episodes": config["bc"]["demo_episodes"],
    "evaluation_episodes": config["evaluation"]["episodes"],
}, indent=2))


{
  "config_name": "smoke_test",
  "training_total_timesteps": 150,
  "bc_demo_episodes": 8,
  "evaluation_episodes": 2
}


## 1. Reward Helper Checks

These are deterministic checks against small handcrafted arrays. The goal is to verify the helper functions directly before involving the full environment.


In [3]:
reward_config = RewardConfig(turn_penalty=-1.0, assassin_penalty=-25.0, shaped_weight=1.0)

achieved = np.array([[1, 0, 1], [1, 1, 0]], dtype=np.float32)
desired = np.array([[1, 0, 0], [1, 1, 1]], dtype=np.float32)
subset_result = subset_achieved(achieved, desired)
assert subset_result.tolist() == [True, False]

clue_emb = np.array([1.0, 0.0], dtype=np.float32)
target_embs = np.array([[1.0, 0.0], [0.8, 0.2]], dtype=np.float32)
bad_embs = np.array([[0.0, 1.0], [0.2, 0.8]], dtype=np.float32)
margin = clue_margin(clue_emb, target_embs, bad_embs)
assert margin > 0.0

breakdown = build_step_reward(
    achieved_goal=np.array([1, 0, 1], dtype=np.float32),
    desired_goal=np.array([1, 0, 0], dtype=np.float32),
    reward_without_goal=2.5,
    clue_margin_value=float(margin),
    assassin_hit=False,
    config=reward_config,
)
assert breakdown.goal_achieved is True
assert breakdown.total == 2.5

single_reward = compute_goal_conditioned_reward(
    np.array([1, 0, 1], dtype=np.float32),
    np.array([1, 1, 0], dtype=np.float32),
    {"reward_without_goal": 2.5},
    reward_config,
)
vector_reward = compute_goal_conditioned_reward(
    achieved,
    desired,
    [{"reward_without_goal": 1.0}, {"reward_without_goal": 2.0}],
    reward_config,
)
assert float(single_reward) == 1.5
assert np.allclose(vector_reward, np.array([1.0, 1.0], dtype=np.float32))

pd.DataFrame([
    {
        "subset_achieved": subset_result.tolist(),
        "clue_margin": float(margin),
        "breakdown_total": breakdown.total,
        "single_goal_reward": float(single_reward),
        "vector_goal_reward": vector_reward.tolist(),
    }
])


,subset_achieved,clue_margin,breakdown_total,single_goal_reward,vector_goal_reward
0,"[True, False]",0.727607,2.5,1.5,"[1.0, 1.0]"


## 2. Evaluation Helper Checks

This section verifies metric summarization and the two supported agent action-selection interfaces.


In [4]:
summary = summarize_metrics([
    EpisodeMetrics(episode_return=1.5, turns=2, won=True, assassin_hit=False, friendly_revealed=3, friendly_total=4),
    EpisodeMetrics(episode_return=-0.5, turns=4, won=False, assassin_hit=True, friendly_revealed=1, friendly_total=4),
])
assert summary["mean_return"] == 0.5
assert summary["mean_turns"] == 3.0
assert summary["win_rate"] == 0.5
assert summary["assassin_rate"] == 0.5
assert summary["friendly_reveal_rate"] == 0.5

class PredictAgent:
    def predict(self, obs, deterministic=True):
        return np.array([42], dtype=np.float32), None

class SelectAgent:
    def select_action(self, env):
        return np.array([7], dtype=np.float32)

assert select_policy_action(PredictAgent(), {"x": 1}, None).tolist() == [42.0]
assert select_policy_action(SelectAgent(), {"x": 1}, object()).tolist() == [7.0]

class TinyEnv:
    def __init__(self):
        self.board_config = type("Cfg", (), {"num_friendly": 3})()
        self.remaining_friendly_indices = [1]
        self.steps = 0

    def reset(self):
        self.steps = 0
        return {"obs": 0}, {}

    def step(self, action):
        self.steps += 1
        done = self.steps >= 2
        info = {
            "assassin_hit": self.steps == 2,
            "won": self.steps == 1,
        }
        return {"obs": self.steps}, 1.25, done, False, info

eval_result = evaluate_agent(PredictAgent(), TinyEnv, episodes=3)
assert eval_result["episodes"] == 3
assert eval_result["mean_turns"] == 2.0

pd.DataFrame([
    {k: eval_result[k] for k in ["mean_return", "mean_turns", "win_rate", "assassin_rate", "friendly_reveal_rate", "episodes"]}
])


,mean_return,mean_turns,win_rate,assassin_rate,friendly_reveal_rate,episodes
0,2.5,2.0,1.0,1.0,0.666667,3


## 3. Real Environment and SAC Agent Smoke Test

This is the first point where we touch the actual environment, embeddings, demonstrations, replay buffer, and SAC wrapper.


In [5]:
embedding_store = build_embedding_store(config)
env_factory = build_env_factory(config, embedding_store)
env = env_factory()
obs, reset_info = env.reset()

assert set(obs.keys()) == {"observation", "achieved_goal", "desired_goal"}
assert env.board_config.board_size == 25
assert len(env.legal_clue_indices) > 0

demos = generate_demonstrations(env_factory=env_factory, num_episodes=2, max_steps_per_episode=2)
assert len(demos) > 0

training_cfg = config["training"]
sac_config = SACTrainingConfig(
    learning_rate=training_cfg["learning_rate"],
    buffer_size=training_cfg["buffer_size"],
    learning_starts=training_cfg["learning_starts"],
    batch_size=training_cfg["batch_size"],
    gamma=training_cfg["gamma"],
    tau=training_cfg["tau"],
    total_timesteps=20,
    policy_kwargs={"net_arch": [64, 64]},
    n_sampled_goal=training_cfg["n_sampled_goal"],
    goal_selection_strategy=training_cfg["goal_selection_strategy"],
)

agent = SACSpymasterAgent(env, config=sac_config, use_her=training_cfg["use_her"])
bc_losses = agent.bc_pretrain(demos, epochs=1, batch_size=2, learning_rate=1e-3)
assert len(bc_losses) == 1

agent.seed_replay_buffer(demos)
assert agent.model.replay_buffer.size() >= len(demos)

training_summary = agent.learn(
    total_timesteps=20,
    demos=demos,
    bc_epochs=1,
    bc_batch_size=2,
    bc_learning_rate=1e-3,
    seed_buffer=True,
)
action, _ = agent.predict(obs, deterministic=True)
assert action.shape == (env.embedding_store.dimension + 1,)

pd.DataFrame([
    {
        "board_size": env.board_config.board_size,
        "legal_clues": len(env.legal_clue_indices),
        "demo_count": len(demos),
        "bc_losses": training_summary["bc_losses"],
        "pred_action_dim": int(action.shape[0]),
    }
])


,board_size,legal_clues,demo_count,bc_losses,pred_action_dim
0,25,2000,4,[0.3900570571422577],301


## 4. Full Smoke Training Pipeline

This runs the same training entry point used by the repo, but writes artifacts to a separate notebook-specific directory so the default logs stay untouched.


In [6]:
notebook_config = deepcopy(config)
notebook_config["name"] = "phase2_notebook_smoke"
notebook_config["output"] = deepcopy(config["output"])
notebook_config["output"]["output_dir"] = "experiments/notebook_phase2_logs"
notebook_config["output"]["metrics_file"] = "phase2_training_metrics.json"
notebook_config["output"]["model_file"] = "phase2_sac_her_model.zip"

results = run_training_pipeline(notebook_config)
metrics_path = PROJECT_ROOT / notebook_config["output"]["output_dir"] / notebook_config["output"]["metrics_file"]
model_path = PROJECT_ROOT / notebook_config["output"]["output_dir"] / notebook_config["output"]["model_file"]

assert results["config_name"] == "phase2_notebook_smoke"
assert results["demo_transitions"] > 0
assert results["sac_metrics"]["episodes"] == notebook_config["evaluation"]["episodes"]
assert results["greedy_metrics"]["episodes"] == notebook_config["evaluation"]["episodes"]
assert metrics_path.exists()
assert model_path.exists()

pd.DataFrame([
    {
        "config_name": results["config_name"],
        "demo_transitions": results["demo_transitions"],
        "sac_mean_return": results["sac_metrics"]["mean_return"],
        "sac_win_rate": results["sac_metrics"]["win_rate"],
        "greedy_mean_return": results["greedy_metrics"]["mean_return"],
        "greedy_win_rate": results["greedy_metrics"]["win_rate"],
        "metrics_path": str(metrics_path.relative_to(PROJECT_ROOT)),
        "model_path": str(model_path.relative_to(PROJECT_ROOT)),
    }
])


,config_name,demo_transitions,sac_mean_return,sac_win_rate,greedy_mean_return,greedy_win_rate,metrics_path,model_path
0,phase2_notebook_smoke,64,-22.370952,0.0,-7.546839,1.0,experiments/notebook_phase2_logs/phase2_training_metrics.json,experiments/notebook_phase2_logs/phase2_sac_her_model.zip
